# 05 — Tune CatBoost, blend, and search per-class thresholds
1. Bounded staged tuning of CatBoost (depth, learning rate, l2, class weights).
2. Convex soft-vote blend of the model OOF matrices (weights searched on OOF).
3. Coordinate-ascent per-class probability multipliers on the **blended** OOF.
Everything is decided on OOF (never the public LB).

In [1]:
# --- Shared toolbox (identical across notebooks; see build_notebooks.py) ---
import warnings, json
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED = 42
N_FOLDS = 5
CLASS_NAMES = {0: "Wake", 1: "Light", 2: "Deep", 3: "REM"}
CLASSES = np.array([0, 1, 2, 3])
EOG = "eog_burst_index"            # the only column with missing values (~50%)

RAW_FEATURES = [
    "eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
    "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power", "eeg_spectral_entropy",
    "eeg_spindle_density", "eeg_kcomplex_rate", "emg_chin_tone", "emg_tone_variance",
    "eog_movement_density", "eog_amplitude", "heart_rate_mean", "heart_rate_variability",
    "respiration_rate", "respiration_variability", "spo2_mean", "body_movement_index",
    EOG,
]
POWER = ["eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
         "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power"]

HERE = Path.cwd()
ART = HERE / "artifacts"; ART.mkdir(exist_ok=True)
SUB = HERE / "submissions"; SUB.mkdir(exist_ok=True)


def load_data():
    """Return (train_df, test_df). Features kept as-is (NaN preserved)."""
    tr = pd.read_csv(HERE / "train.csv")
    te = pd.read_csv(HERE / "test.csv")
    return tr, te


def macro_f1(y_true, y_pred):
    """Competition metric: macro-averaged F1 over the 4 classes."""
    return f1_score(y_true, y_pred, average="macro")


def per_class_f1(y_true, y_pred):
    f = f1_score(y_true, y_pred, average=None, labels=CLASSES)
    return {CLASS_NAMES[c]: round(float(f[i]), 4) for i, c in enumerate(CLASSES)}


def softplus(x):
    """Numerically stable log(1+exp(x)); strictly positive and monotonic.
    Used to turn z-scored band powers (~50% negative) into positive magnitudes
    so band ratios are well-defined instead of dividing by near-zero."""
    x = np.asarray(x, dtype=float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def _aligned_proba(model, X):
    """predict_proba with columns aligned to CLASSES = [0,1,2,3]."""
    p = model.predict_proba(X)
    cls = list(np.asarray(model.classes_))
    idx = [cls.index(c) for c in CLASSES]
    return p[:, idx]


def run_oof(make_model, X, y, X_test, folds, needs_impute=False, use_eval_set=False):
    """Out-of-fold training under fixed folds.

    Returns (oof, test_p, oof_macro, fold_scores):
      oof     : (n_train, 4) out-of-fold probabilities (each row predicted once)
      test_p  : (n_test, 4) test probabilities, averaged over the 5 fold-models
      oof_macro: global macro-F1 over the assembled OOF matrix (primary metric)

    Two model families, identical folds:
      - CatBoost (needs_impute=False): NaN passed through natively.
      - sklearn trees (needs_impute=True): add EOG-missing flag + fill EOG NaN
        with the TRAIN-FOLD median (fit on train fold only -> no leakage)."""
    n = len(y)
    oof = np.zeros((n, len(CLASSES)))
    test_p = np.zeros((len(X_test), len(CLASSES)))
    fold_scores = []
    for tr_idx, va_idx in folds:
        Xtr, Xva, Xte = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy(), X_test.copy()
        ytr, yva = y[tr_idx], y[va_idx]
        if needs_impute:
            med = Xtr[EOG].median()
            for d in (Xtr, Xva, Xte):
                if EOG + "_missing" not in d.columns:
                    d[EOG + "_missing"] = d[EOG].isna().astype("int8")
                d[EOG] = d[EOG].fillna(med)
            assert not Xtr.isna().any().any(), "NaN remained after impute"
        model = make_model()
        if use_eval_set:
            model.fit(Xtr, ytr, eval_set=(Xva, yva))
        else:
            model.fit(Xtr, ytr)
        oof[va_idx] = _aligned_proba(model, Xva)
        test_p += _aligned_proba(model, Xte) / len(folds)
        fold_scores.append(macro_f1(yva, oof[va_idx].argmax(1)))
    oof_macro = macro_f1(y, oof.argmax(1))
    return oof, test_p, oof_macro, fold_scores


def load_folds():
    """Load the fixed StratifiedKFold split saved by 02_baseline."""
    d = np.load(ART / "folds.npz", allow_pickle=True)
    return [(d[f"tr{i}"], d[f"va{i}"]) for i in range(N_FOLDS)]


In [2]:
from catboost import CatBoostClassifier
cols = json.load(open(ART / "feature_cols.json"))["columns"]
Xtr = pd.DataFrame(np.load(ART / "features_v1_train.npy"), columns=cols)
Xte = pd.DataFrame(np.load(ART / "features_v1_test.npy"), columns=cols)
y = np.load(ART / "y_train.npy")
folds = load_folds()

## 1. Bounded staged CatBoost tuning
We sweep one hyperparameter at a time around a sensible default (≈13 fits of 5 folds, not a full grid). Iterations are capped with early stopping, never hand-tuned. All seeds fixed.

In [3]:
def make_cat(depth=6, lr=0.03, l2=3.0, balanced=False):
    kw = dict(loss_function="MultiClass", eval_metric="TotalF1:average=Macro",
        iterations=3000, learning_rate=lr, depth=depth, l2_leaf_reg=l2,
        random_seed=SEED, od_type="Iter", od_wait=150, use_best_model=True,
        thread_count=-1, allow_writing_files=False, verbose=False)
    if balanced:
        kw["auto_class_weights"] = "Balanced"
    return CatBoostClassifier(**kw)

def cat_score(**kw):
    _, _, m, _ = run_oof(lambda: make_cat(**kw), Xtr, y, Xte, folds,
                         needs_impute=False, use_eval_set=True)
    return m

best = {"depth": 6, "lr": 0.03, "l2": 3.0, "balanced": False}
best_score = cat_score(**best)
print(f"start depth=6 lr=0.03 l2=3 -> {best_score:.5f}")
for depth in [5, 7, 8]:
    s = cat_score(**{**best, "depth": depth})
    print(f"  depth={depth}: {s:.5f}")
    if s > best_score: best_score, best["depth"] = s, depth
for lr in [0.02, 0.05]:
    s = cat_score(**{**best, "lr": lr})
    print(f"  lr={lr}: {s:.5f}")
    if s > best_score: best_score, best["lr"] = s, lr
for l2 in [1.0, 5.0, 9.0]:
    s = cat_score(**{**best, "l2": l2})
    print(f"  l2={l2}: {s:.5f}")
    if s > best_score: best_score, best["l2"] = s, l2
for bal in [True]:
    s = cat_score(**{**best, "balanced": bal})
    print(f"  balanced={bal}: {s:.5f}")
    if s > best_score: best_score, best["balanced"] = s, bal
print("\nbest config:", best, "-> OOF macro-F1", round(best_score, 5))

start depth=6 lr=0.03 l2=3 -> 0.82898


  depth=5: 0.82152


  depth=7: 0.82678


  depth=8: 0.82573


  lr=0.02: 0.82427


  lr=0.05: 0.82446


  l2=1.0: 0.82549


  l2=5.0: 0.82756


  l2=9.0: 0.82530


  balanced=True: 0.82499

best config: {'depth': 6, 'lr': 0.03, 'l2': 3.0, 'balanced': False} -> OOF macro-F1 0.82898


In [4]:
# Refit the tuned CatBoost across folds; save its OOF + test matrices
cat_oof, cat_test, cat_macro, _ = run_oof(lambda: make_cat(**best), Xtr, y, Xte,
    folds, needs_impute=False, use_eval_set=True)
np.save(ART / "catboost_tuned_oof.npy", cat_oof)
np.save(ART / "catboost_tuned_test.npy", cat_test)
json.dump(best, open(ART / "catboost_best_params.json", "w"), indent=2)
print("tuned CatBoost OOF macro-F1:", round(cat_macro, 5))

tuned CatBoost OOF macro-F1: 0.82898


## 2. Convex blend (weights searched on OOF)
Blend CatBoost(tuned) + RF + ET + GB. We search nonnegative weights summing to 1 on a 0.1 grid; weak members get ~0 weight.

In [5]:
members = ["catboost_tuned", "rf", "et", "gb"]
oofs = {m: (cat_oof if m == "catboost_tuned" else np.load(ART / f"{m}_oof.npy")) for m in members}
tests = {m: (cat_test if m == "catboost_tuned" else np.load(ART / f"{m}_test.npy")) for m in members}

def compositions(total, parts):
    if parts == 1:
        yield (total,); return
    for i in range(total + 1):
        for rest in compositions(total - i, parts - 1):
            yield (i,) + rest

oof_list = [oofs[m] for m in members]
STEPS = 10
best_w, best_blend = None, -1
for comp in compositions(STEPS, len(members)):
    w = np.array(comp) / STEPS
    blended = sum(wi * o for wi, o in zip(w, oof_list))
    s = macro_f1(y, blended.argmax(1))
    if s > best_blend:
        best_blend, best_w = s, w
weights = {m: round(float(w), 2) for m, w in zip(members, best_w)}
print("best blend weights:", weights, "-> OOF macro-F1", round(best_blend, 5))

blend_oof = sum(best_w[i] * oofs[m] for i, m in enumerate(members))
blend_test = sum(best_w[i] * tests[m] for i, m in enumerate(members))
np.save(ART / "blend_oof.npy", blend_oof)
np.save(ART / "blend_test.npy", blend_test)
json.dump({"members": members, "weights": weights, "oof_macro": round(float(best_blend), 5)},
          open(ART / "blend.json", "w"), indent=2)

best blend weights: {'catboost_tuned': 1.0, 'rf': 0.0, 'et': 0.0, 'gb': 0.0} -> OOF macro-F1 0.82898


## 3. Per-class threshold (probability multiplier) search
Coordinate ascent on the blended OOF. Multiplying class probabilities then argmax shifts per-class priors to lift the weakest class — exactly what macro-F1 rewards. Searched on OOF only.

In [6]:
def search_thresholds(oof, y, grid, passes=12):
    m = np.ones(oof.shape[1])
    best = macro_f1(y, oof.argmax(1))
    for _ in range(passes):
        improved = False
        for c in range(len(m)):
            keep_g, keep_s = m[c], best
            for g in grid:
                m[c] = g; s = macro_f1(y, (oof * m).argmax(1))
                if s > keep_s + 1e-12: keep_s, keep_g = s, g
            m[c] = keep_g
            if keep_s > best + 1e-12: best, improved = keep_s, True
        if not improved: break
    return m, best

# single thorough coordinate-ascent pass (fine 0.02 grid over a wide range)
mult_final, final = search_thresholds(blend_oof, y, np.round(np.arange(0.4, 1.61, 0.02), 3))
print("no-threshold blended OOF macro-F1:", round(macro_f1(y, blend_oof.argmax(1)), 5))
print("multipliers:", np.round(mult_final, 3), "-> OOF macro-F1", round(final, 5))
print("per-class F1 before:", per_class_f1(y, blend_oof.argmax(1)))
print("per-class F1 after :", per_class_f1(y, (blend_oof * mult_final).argmax(1)))
json.dump({"multipliers": [round(float(x), 3) for x in mult_final],
           "oof_macro": round(float(final), 5)},
          open(ART / "thresholds.json", "w"), indent=2)

no-threshold blended OOF macro-F1: 0.82898
multipliers: [1. 1. 1. 1.] -> OOF macro-F1 0.82898
per-class F1 before: {'Wake': 0.8516, 'Light': 0.8507, 'Deep': 0.7766, 'REM': 0.8369}
per-class F1 after : {'Wake': 0.8516, 'Light': 0.8507, 'Deep': 0.7766, 'REM': 0.8369}


In [7]:
print("confusion BEFORE thresholds:"); print(confusion_matrix(y, blend_oof.argmax(1)))
print("confusion AFTER thresholds:");  print(confusion_matrix(y, (blend_oof * mult_final).argmax(1)))

confusion BEFORE thresholds:
[[1690   69  156   86]
 [  60 2106  158  118]
 [ 137  194 1709  197]
 [  81  140  141 1958]]
confusion AFTER thresholds:
[[1690   69  156   86]
 [  60 2106  158  118]
 [ 137  194 1709  197]
 [  81  140  141 1958]]
